# Linear Regression Implementation
The start of every machine learning project begins the same way:
- Data Inspection
- Data Visualization
- Data Regularization
- Model Implementation

Our data is the [California Housing Dataset](https://www.kaggle.com/datasets/camnugent/california-housing-prices) from Kaggle. It contains 20,640 block groups from the 1990 California census. Each block group has features like median income, house age, average rooms, population, and geographic coordinates. We will predict the **median house value** for each block group.

---

## Phase 1: Data Inspection
We will first load a dataframe and inspect the data to understand its structure, types, and any missing values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn import linear_model

data_df = pd.read_csv("housing.csv")

print("Shape is: ", data_df.shape)
print("\nColumn types:")
print(data_df.dtypes)
print("\nMissing values:")
print(data_df.isnull().sum())
data_df.head(10)

We have 20,640 samples with 9 numerical features and 1 categorical feature (`ocean_proximity`). There are 207 missing values in `total_bedrooms` which we will need to handle. The target variable is `median_house_value`.

Let's look at the basic statistics to understand the scale and distribution of each feature.

In [ ]:
data_df.describe()

A few things stand out:
- **Median income** ranges from ~0.5 to 15 (in tens of thousands of dollars)
- **Median house value** ranges from \$14,999 to \$500,001 (the cap at 500,001 suggests truncation)
- **Total rooms** and **population** have very large ranges, indicating significant variance across block groups
- Feature scales differ enormously (e.g., median_income ~0-15 vs total_rooms ~2-39,000), so normalization will be essential

---

## Phase 2: Data Visualization
The dataset has 8 numerical features and 1 output. We cannot visualize 8-dimensional data, but we can inspect each feature against the target to identify correlations.

In [ ]:
numerical_features = data_df.select_dtypes(include=[np.number]).columns.drop('median_house_value')

fig, axs = plt.subplots(4, 2, figsize=(20, 24))
fig.suptitle("Feature vs Median House Value", fontsize=24)

for idx, feature in enumerate(numerical_features):
    row, col = idx // 2, idx % 2
    axs[row, col].scatter(data_df[feature], data_df['median_house_value'], alpha=0.1, s=5)
    axs[row, col].set_xlabel(feature, fontsize=14)
    axs[row, col].set_ylabel('median_house_value', fontsize=14)

plt.tight_layout()
plt.show()

Key observations from the scatter plots:
- **Median income** shows the strongest positive correlation with house value - this will likely be our most important feature
- **Longitude and latitude** reveal geographic clustering - houses near the coast (specific lat/long bands) tend to be more expensive
- **Housing median age** shows a slight positive trend but with high variance
- The horizontal line at $500,001 confirms the value cap in the dataset
- **Total rooms, bedrooms, population, and households** show weak individual correlations with lots of spread

Let's visualize the geographic distribution of house values, since California geography (coastal vs inland) likely plays a major role in pricing.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(data_df['longitude'], data_df['latitude'],
                     c=data_df['median_house_value'], cmap='magma',
                     alpha=0.4, s=10)
plt.colorbar(scatter, label='Median House Value ($)')
ax.set_xlabel('Longitude', fontsize=14)
ax.set_ylabel('Latitude', fontsize=14)
ax.set_title('California Housing Prices by Location', fontsize=18)
plt.tight_layout()
plt.show()

This is effectively a map of California. The brightest (most expensive) areas correspond to the San Francisco Bay Area and coastal Los Angeles/San Diego. Inland areas are consistently cheaper. This confirms that geography is a strong predictor of house value.

Let's also look at the categorical feature `ocean_proximity`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 6))

data_df['ocean_proximity'].value_counts().plot(kind='bar', ax=axs[0], color='steelblue')
axs[0].set_title('Ocean Proximity Distribution', fontsize=16)
axs[0].set_ylabel('Count', fontsize=14)
axs[0].tick_params(axis='x', rotation=45)

data_df.boxplot(column='median_house_value', by='ocean_proximity', ax=axs[1])
axs[1].set_title('House Value by Ocean Proximity', fontsize=16)
axs[1].set_ylabel('Median House Value ($)', fontsize=14)
axs[1].set_xlabel('Ocean Proximity', fontsize=14)
plt.suptitle('')

plt.tight_layout()
plt.show()

The boxplot confirms that `ISLAND` properties are the most expensive (though rare - only 5 samples), while `INLAND` properties are consistently the cheapest. `<1H OCEAN` and `NEAR BAY` are in the middle-to-upper range. This categorical feature carries useful information.

---

## Phase 3: Data Regularization

Now we need to prepare the data for our model. This involves:
1. Handling missing values in `total_bedrooms` (207 nulls)
2. Encoding the categorical `ocean_proximity` feature as numeric
3. Normalizing all features using z-score standardization

We regularize the data using:
$
\Large Z = \dfrac{X - \mu}{\sigma}
$

This reduces the spread of each feature to a common scale so no single feature dominates due to its magnitude.

In [ ]:
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())

data_encoded = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

print("Encoded shape:", data_encoded.shape)
print("\nNew columns:", data_encoded.columns.tolist())
data_encoded.head()

In [ ]:
data = data_encoded.to_numpy()
target_col = data_encoded.columns.get_loc('median_house_value')

feature_cols = [i for i in range(data.shape[1]) if i != target_col]
x_data = data[:, feature_cols]
y = data[:, target_col]

feature_names = [data_encoded.columns[i] for i in feature_cols]
print(f"Features ({len(feature_names)}): {feature_names}")
print(f"Target: median_house_value")

mu = x_data.mean(axis=0)
sigma = x_data.std(axis=0)
sigma[sigma == 0] = 1

r_data = (x_data - mu) / sigma
r_data = np.c_[np.ones(r_data.shape[0]), r_data]
print(f"\nRegularized data shape (with bias): {r_data.shape}")

We filled 207 missing `total_bedrooms` values with the median (a robust choice against outliers), one-hot encoded `ocean_proximity` into 5 binary columns, and standardized all features to mean=0, std=1. We also appended a bias column of ones so our weight vector can learn an intercept term.

---

## Splitting Data

We partition the data so we can train on a subset, then test with data the model has never seen. This tells us whether the model generalizes or just memorizes training data. We will use a 67/33 train/test split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    r_data, y, test_size=0.33, shuffle=True, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

---

## Phase 4: Train the Model

We will train our linear regression model using two methods:

### Gradient Descent
An iterative algorithm that updates weights to minimize error. A prediction is calculated as $\hat{y} = X\theta$, and the gradient is:

$\nabla\theta = \frac{2}{n} X^T(\hat{y} - y)$

Weights are updated: $\theta_{i} = \theta_{i-1} - \alpha \nabla\theta$

The learning rate $\alpha$ controls step size - too large and we overshoot, too small and convergence is slow.

### Normal Equation
The optimal weights can be computed directly:

$\hat{\theta} = (X^TX)^{-1}(X^Ty)$

This gives the exact solution in one step, no iteration needed.

In [ ]:
gMSE_all = []
nMSE_all = []
i_all = []

theta = np.random.randn(X_train.shape[1]) * np.sqrt(2)

theta_hat = np.linalg.pinv(X_train.T.dot(X_train)).dot(X_train.T).dot(y_train)

alpha = 0.05

for i in range(1000):
    y_hat_gd = X_train @ theta
    gMSE = (1 / X_train.shape[0]) * np.sum((y_hat_gd - y_train) ** 2)
    gMSE_all.append(gMSE)
    i_all.append(i)

    delta = (2 / X_train.shape[0]) * (X_train.T @ (y_hat_gd - y_train))
    theta = theta - alpha * delta

    y_hat_ne = X_train @ theta_hat
    nMSE = (1 / X_train.shape[0]) * np.sum((y_hat_ne - y_train) ** 2)
    nMSE_all.append(nMSE)

gy_hat = X_test @ theta
ny_hat = X_test @ theta_hat

gMSE_test = (1 / X_test.shape[0]) * np.sum((gy_hat - y_test) ** 2)
nMSE_test = (1 / X_test.shape[0]) * np.sum((ny_hat - y_test) ** 2)

fig, ax = plt.subplots(1, 2, figsize=(20, 5))

ax[0].plot(i_all[:200], gMSE_all[:200], label="Gradient Descent")
ax[0].plot(i_all[:200], nMSE_all[:200], label="Normal Equation")
ax[0].set_xlabel("Iteration", fontsize=14)
ax[0].set_ylabel("MSE", fontsize=14)
ax[0].set_title("Training MSE vs Iteration", fontsize=16)
ax[0].legend(fontsize=12)

ax[1].scatter(range(X_test.shape[0]), gy_hat - y_test, label="Gradient Descent Error", s=5, alpha=0.3)
ax[1].scatter(range(X_test.shape[0]), ny_hat - y_test, label="Normal Equation Error", s=5, alpha=0.3, c='goldenrod')
ax[1].set_xlabel("Sample Number", fontsize=14)
ax[1].set_ylabel("Error ($)", fontsize=14)
ax[1].set_title("Test Error vs Sample Number", fontsize=16)
ax[1].legend(fontsize=12)

text = f"Test MSE:\nGradient Descent: {round(gMSE_test, 2)}\nNormal Eq: {round(nMSE_test, 2)}"
ax[1].text(0.02, 0.95, text, transform=ax[1].transAxes, verticalalignment='top',
           fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

The left plot shows gradient descent converging toward the normal equation solution over ~100 iterations. The normal equation instantly finds the optimal weights, while gradient descent iteratively walks downhill.

The right plot shows per-sample errors on the test set. Both methods produce very similar errors once gradient descent has converged. The errors are centered around zero, meaning the model is not systematically over- or under-predicting, though some samples have large errors (outliers or unusual properties).

---

## Phase 5: Results Visualization

Now that our model is trained, let's see how well our predictions align with the actual values. We'll plot each input feature against both the true and predicted house values.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
X_test_1d = pca.fit_transform(X_test[:, 1:])

sort_idx = X_test_1d.flatten().argsort()

fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(X_test_1d[sort_idx], y_test[sort_idx], alpha=0.2, s=8, label='Actual', color='steelblue')
ax.scatter(X_test_1d[sort_idx], ny_hat[sort_idx], alpha=0.2, s=8, label='Predicted', color='coral')
ax.set_xlabel('PCA Component 1', fontsize=14)
ax.set_ylabel('Median House Value ($)', fontsize=14)
ax.set_title('Predicted vs Actual House Values (PCA Projection)', fontsize=16)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plot_features = ['median_income', 'housing_median_age', 'latitude', 'longitude']
plot_indices = [feature_names.index(f) + 1 for f in plot_features]

fig, axs = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle("Predicted vs Actual by Feature", fontsize=24)

for idx, (feat, col_idx) in enumerate(zip(plot_features, plot_indices)):
    row, col = idx // 2, idx % 2
    axs[row, col].scatter(X_test[:, col_idx], y_test, alpha=0.15, s=5, label='Actual')
    axs[row, col].scatter(X_test[:, col_idx], ny_hat, alpha=0.15, s=5, label='Predicted', color='coral')
    axs[row, col].set_xlabel(feat, fontsize=14)
    axs[row, col].set_ylabel('median_house_value', fontsize=14)
    axs[row, col].legend(fontsize=12)

plt.tight_layout()
plt.show()

Looking at the feature-level plots:
- **Median income** shows the tightest alignment between predicted and actual - the model captures this relationship well
- **Housing median age** - the model captures the slight upward trend but misses the variance
- **Latitude and longitude** - predictions follow the general geographic price gradient but can't capture the nonlinear coastal premium

The predictions are concentrated in the bulk of the data. Outliers and extreme values (especially the $500,001 cap) create errors that linear regression cannot handle well.

Let's also visualize predictions geographically.

In [ ]:
long_idx = feature_names.index('longitude') + 1
lat_idx = feature_names.index('latitude') + 1

fig = plt.figure(figsize=(16, 10))
ax = fig.add_subplot(projection='3d')

ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], ny_hat,
             c=ny_hat, cmap='Reds', alpha=0.3, s=5, label='Predicted')
ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], y_test,
             c=y_test, cmap='Greens', alpha=0.3, s=5, label='Actual')

ax.set_xlabel('Longitude (normalized)')
ax.set_ylabel('Latitude (normalized)')
ax.set_zlabel('Median House Value ($)')
ax.set_title('House Value vs Location', fontsize=18)
ax.legend(fontsize=14)
ax.view_init(20, 30)
plt.tight_layout()
plt.show()